# Data loading

In [2]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# Clarans predefined

In [7]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    silhouette_score, 
    davies_bouldin_score, 
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    fowlkes_mallows_score
)
from sklearn.decomposition import PCA
from sklearn_extra.cluster import KMedoids
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Get the Code directory (project root)
current_dir = Path.cwd()
code_dir = current_dir.parent.parent.parent.parent
print(f"Code directory: {code_dir}")

# Define all data paths
PATHS = {
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

class CLARA:
    """CLARA (Clustering Large Applications) - Memory-efficient K-Medoids"""
    def __init__(self, n_clusters, n_sampling=5, sample_size=1000, random_state=42):
        self.n_clusters = n_clusters
        self.n_sampling = n_sampling
        self.sample_size = sample_size
        self.random_state = random_state
        self.best_medoid_indices_ = None
        self.labels_ = None
        self.inertia_ = None
        
    def fit(self, X):
        """Fit CLARA algorithm"""
        np.random.seed(self.random_state)
        best_cost = np.inf
        
        # Convert to numpy array and ensure float64
        if isinstance(X, pd.DataFrame):
            X_array = X.values.astype(np.float64)
        else:
            X_array = np.asarray(X, dtype=np.float64)
        
        print(f"\nRunning CLARA with {self.n_sampling} samplings of {self.sample_size} samples each...")
        
        for i in range(self.n_sampling):
            # Sample data
            sample_idx = np.random.choice(len(X_array), 
                                         min(self.sample_size, len(X_array)), 
                                         replace=False)
            X_sample = X_array[sample_idx]
            
            # Run K-Medoids on sample
            kmedoids = KMedoids(
                n_clusters=self.n_clusters,
                metric='euclidean',
                method='pam',
                init='k-medoids++',
                max_iter=300,
                random_state=self.random_state + i
            )
            kmedoids.fit(X_sample)
            
            # Get medoids from sample
            sample_medoid_indices = kmedoids.medoid_indices_
            medoids = X_sample[sample_medoid_indices].astype(np.float64)
            
            # Assign all data points to nearest medoid using chunked processing
            labels = np.zeros(len(X_array), dtype=np.int32)
            min_distances = np.full(len(X_array), np.inf, dtype=np.float64)
            
            chunk_size = 5000
            for start_idx in range(0, len(X_array), chunk_size):
                end_idx = min(start_idx + chunk_size, len(X_array))
                X_chunk = X_array[start_idx:end_idx]
                
                # Calculate distances for this chunk
                chunk_distances = np.zeros((len(X_chunk), self.n_clusters), dtype=np.float64)
                for j, medoid in enumerate(medoids):
                    diff = X_chunk - medoid
                    chunk_distances[:, j] = np.sqrt(np.sum(diff ** 2, axis=1))
                
                labels[start_idx:end_idx] = np.argmin(chunk_distances, axis=1)
                min_distances[start_idx:end_idx] = np.min(chunk_distances, axis=1)
            
            cost = np.sum(min_distances)
            
            # Update best solution
            if cost < best_cost:
                best_cost = cost
                # Find actual indices of medoids in full dataset
                self.best_medoid_indices_ = []
                for medoid in medoids:
                    # Find closest point in full dataset to this medoid
                    diff = X_array - medoid
                    distances_to_medoid = np.sqrt(np.sum(diff ** 2, axis=1))
                    self.best_medoid_indices_.append(np.argmin(distances_to_medoid))
                self.labels_ = labels.copy()
                self.inertia_ = cost
            
            print(f"  Sampling {i+1}/{self.n_sampling}: cost = {cost:.2f}")
        
        self.best_medoid_indices_ = np.array(self.best_medoid_indices_)
        print(f"Best cost: {best_cost:.2f}")
        return self
    
    def fit_predict(self, X):
        """Fit and return labels"""
        self.fit(X)
        return self.labels_
    
    @property
    def medoid_indices_(self):
        return self.best_medoid_indices_

def load_all_data():
    """Load all data files"""
    data = {}
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
    
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
    
    return data

def find_optimal_k(X, y_true, k_range=range(2, 5), max_samples=3000):
    """Find optimal number of clusters using various metrics"""
    print("\n" + "="*50)
    print(f"FINDING OPTIMAL K (using CLARA)")
    print("="*50)
    
    # Sample data for faster computation
    if len(X) > max_samples:
        sample_idx = np.random.choice(len(X), max_samples, replace=False)
        X_sample = X.iloc[sample_idx] if isinstance(X, pd.DataFrame) else X[sample_idx]
        y_sample = y_true.iloc[sample_idx] if isinstance(y_true, pd.Series) else y_true[sample_idx]
    else:
        X_sample = X
        y_sample = y_true
    
    metrics = {
        'k': [],
        'silhouette': [],
        'davies_bouldin': [],
        'calinski_harabasz': [],
        'inertia': [],
        'ari': [],
        'nmi': [],
        'fmi': []
    }
    
    for k in k_range:
        print(f"\nTesting k={k}...")
        
        # Use CLARA for large datasets
        if len(X_sample) > 5000:
            clara = CLARA(n_clusters=k, n_sampling=3, sample_size=1000, random_state=42)
            labels = clara.fit_predict(X_sample)
            inertia = clara.inertia_
        else:
            # Use regular K-Medoids for smaller samples
            kmedoids = KMedoids(
                n_clusters=k,
                metric='euclidean',
                method='pam',
                init='k-medoids++',
                max_iter=300,
                random_state=42
            )
            labels = kmedoids.fit_predict(X_sample)
            inertia = kmedoids.inertia_
        
        # Calculate metrics
        metrics['k'].append(k)
        metrics['silhouette'].append(silhouette_score(X_sample, labels))
        metrics['davies_bouldin'].append(davies_bouldin_score(X_sample, labels))
        metrics['calinski_harabasz'].append(calinski_harabasz_score(X_sample, labels))
        metrics['inertia'].append(inertia)
        
        # External validation metrics
        metrics['ari'].append(adjusted_rand_score(y_sample, labels))
        metrics['nmi'].append(normalized_mutual_info_score(y_sample, labels))
        metrics['fmi'].append(fowlkes_mallows_score(y_sample, labels))
        
        print(f"  Silhouette: {metrics['silhouette'][-1]:.4f}")
        print(f"  Davies-Bouldin: {metrics['davies_bouldin'][-1]:.4f}")
        print(f"  Calinski-Harabasz: {metrics['calinski_harabasz'][-1]:.4f}")
        print(f"  ARI: {metrics['ari'][-1]:.4f}")
    
    return pd.DataFrame(metrics)

def plot_metrics(metrics_df, save_path=None):
    """Plot clustering metrics for different k values"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('K-Medoids (CLARA) Clustering Metrics vs Number of Clusters', 
                 fontsize=16, fontweight='bold')
    
    plot_configs = [
        ('silhouette', 'Silhouette Score', 'Higher is better', 'green'),
        ('davies_bouldin', 'Davies-Bouldin Index', 'Lower is better', 'red'),
        ('calinski_harabasz', 'Calinski-Harabasz Score', 'Higher is better', 'blue'),
        ('inertia', 'Inertia', 'Lower is better', 'orange'),
        ('ari', 'Adjusted Rand Index', 'Higher is better', 'purple'),
        ('nmi', 'Normalized Mutual Info', 'Higher is better', 'cyan'),
        ('fmi', 'Fowlkes-Mallows Index', 'Higher is better', 'magenta'),
    ]
    
    for idx, (metric, title, subtitle, color) in enumerate(plot_configs):
        ax = axes[idx // 4, idx % 4]
        ax.plot(metrics_df['k'], metrics_df[metric], marker='o', linewidth=2, 
                markersize=8, color=color, label=metric)
        ax.set_xlabel('Number of Clusters (k)', fontsize=10)
        ax.set_ylabel(title, fontsize=10)
        ax.set_title(f'{title}\n({subtitle})', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xticks(metrics_df['k'])
        
        # Mark optimal point
        if 'lower' in subtitle.lower():
            optimal_idx = metrics_df[metric].idxmin()
        else:
            optimal_idx = metrics_df[metric].idxmax()
        optimal_k = metrics_df.loc[optimal_idx, 'k']
        optimal_val = metrics_df.loc[optimal_idx, metric]
        ax.scatter([optimal_k], [optimal_val], color='red', s=200, marker='*', 
                  zorder=5, label=f'Optimal k={optimal_k}')
        ax.legend(loc='best', fontsize=8)
    
    # Remove extra subplot
    fig.delaxes(axes[1, 3])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def perform_clustering(X, y_true, n_clusters, dataset_name='Dataset'):
    """Perform K-Medoids clustering using CLARA for large datasets"""
    print("\n" + "="*50)
    print(f"CLUSTERING: {dataset_name} (k={n_clusters})")
    print("="*50)
    
    # Use CLARA for large datasets
    clara = CLARA(
        n_clusters=n_clusters, 
        n_sampling=5,  # Number of sampling iterations
        sample_size=2000,  # Size of each sample
        random_state=42
    )
    
    labels = clara.fit_predict(X)
    
    # Calculate metrics on full dataset
    print("\nCalculating metrics on full dataset...")
    metrics = {
        'Silhouette Score': silhouette_score(X, labels),
        'Davies-Bouldin Index': davies_bouldin_score(X, labels),
        'Calinski-Harabasz Score': calinski_harabasz_score(X, labels),
        'Inertia': clara.inertia_,
        'Adjusted Rand Index': adjusted_rand_score(y_true, labels),
        'Normalized Mutual Info': normalized_mutual_info_score(y_true, labels),
        'Fowlkes-Mallows Score': fowlkes_mallows_score(y_true, labels)
    }
    
    print("\nClustering Metrics:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    return clara, labels, metrics

def visualize_clusters(X, labels, y_true, medoids_idx, dataset_name='Dataset', save_path=None):
    """Visualize clusters using PCA"""
    print(f"\nGenerating visualizations for {dataset_name}...")
    
    # Sample data for visualization if too large
    if len(X) > 10000:
        print(f"Sampling 10,000 points for visualization...")
        sample_idx = np.random.choice(len(X), 10000, replace=False)
        X_vis = X.iloc[sample_idx] if isinstance(X, pd.DataFrame) else X[sample_idx]
        labels_vis = labels[sample_idx]
        y_vis = y_true.iloc[sample_idx] if isinstance(y_true, pd.Series) else y_true[sample_idx]
    else:
        X_vis = X
        labels_vis = labels
        y_vis = y_true
    
    # Apply PCA
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_vis)
    
    # Transform medoids
    X_array = X.values if isinstance(X, pd.DataFrame) else X
    medoids_pca = pca.transform(X_array[medoids_idx])
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'K-Medoids (CLARA) Clustering Visualization - {dataset_name}', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Predicted clusters
    scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_vis, 
                               cmap='viridis', alpha=0.6, s=30)
    axes[0].scatter(medoids_pca[:, 0], medoids_pca[:, 1], 
                   c='red', marker='X', s=300, edgecolors='black', 
                   linewidths=2, label='Medoids')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    axes[0].set_title('Predicted Clusters', fontsize=12, fontweight='bold')
    axes[0].legend()
    plt.colorbar(scatter1, ax=axes[0], label='Cluster')
    
    # Plot 2: True labels
    scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_vis, 
                               cmap='plasma', alpha=0.6, s=30)
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    axes[1].set_title('True Labels', fontsize=12, fontweight='bold')
    plt.colorbar(scatter2, ax=axes[1], label='Class')
    
    # Plot 3: Cluster sizes
    unique_labels, counts = np.unique(labels, return_counts=True)
    axes[2].bar(unique_labels, counts, color='steelblue', edgecolor='black')
    axes[2].set_xlabel('Cluster', fontsize=11)
    axes[2].set_ylabel('Number of Samples', fontsize=11)
    axes[2].set_title('Cluster Sizes', fontsize=12, fontweight='bold')
    axes[2].set_xticks(unique_labels)
    for i, (label, count) in enumerate(zip(unique_labels, counts)):
        axes[2].text(label, count, str(count), ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def compare_datasets(all_results):
    """Compare clustering results across datasets"""
    print("\n" + "="*50)
    print("COMPARING DATASETS")
    print("="*50)
    
    comparison = pd.DataFrame(all_results).T
    print("\n", comparison)
    
    metrics_to_plot = ['Silhouette Score', 'Davies-Bouldin Index', 
                       'Calinski-Harabasz Score', 'Adjusted Rand Index']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Clustering Performance Comparison Across Datasets', 
                 fontsize=16, fontweight='bold')
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        values = comparison[metric]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        bars = ax.bar(range(len(values)), values, color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xlabel('Dataset', fontsize=11)
        ax.set_ylabel(metric, fontsize=11)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.set_xticks(range(len(values)))
        ax.set_xticklabels(values.index, rotation=15, ha='right')
        ax.grid(axis='y', alpha=0.3)
        
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('clustering_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# Main execution
if __name__ == "__main__":
    data = load_all_data()
    
    if not data:
        print("No data loaded. Exiting.")
        exit()
    
    datasets = [
        ('X_train_smote', 'y_train_smote', 'SMOTE'),
        ('X_train_tomek', 'y_train_tomek', 'Tomek'),
        ('X_train_smote_tomek', 'y_train_smote_tomek', 'SMOTE-Tomek')
    ]
    
    all_results = {}
    
    for X_key, y_key, name in datasets:
        if X_key in data and y_key in data:
            X = data[X_key]
            y = data[y_key]
            
            # Find optimal k
            metrics_df = find_optimal_k(X, y, k_range=range(2, 8))
            plot_metrics(metrics_df, save_path=f'metrics_{name.lower()}.png')
            
            # Use optimal k based on silhouette score
            optimal_k = metrics_df.loc[metrics_df['silhouette'].idxmax(), 'k']
            print(f"\nOptimal k for {name}: {int(optimal_k)}")
            
            # Perform clustering with optimal k
            model, labels, metrics = perform_clustering(X, y, int(optimal_k), name)
            
            # Visualize
            visualize_clusters(X, labels, y, model.medoid_indices_, 
                             name, save_path=f'clusters_{name.lower()}.png')
            
            all_results[name] = metrics
    
    if all_results:
        compare_datasets(all_results)
    
    print("\n" + "="*50)
    print("ANALYSIS COMPLETE")
    print("="*50)

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

LOADING DATA
✓ Loaded X_train_smote: (30570, 64)
✓ Loaded X_train_tomek: (15911, 64)
✓ Loaded X_train_smote_tomek: (30266, 64)
✓ Loaded X_val: (3477, 64)
✓ Loaded X_test: (3477, 64)
✓ Loaded y_train_smote: 30570 samples
✓ Loaded y_train_tomek: 15911 samples
✓ Loaded y_train_smote_tomek: 30266 samples
✓ Loaded y_val: 3477 samples
✓ Loaded y_test: 3477 samples

FINDING OPTIMAL K (using CLARA)

Testing k=2...
  Silhouette: 0.9714
  Davies-Bouldin: 0.0296
  Calinski-Harabasz: 478406.6452
  ARI: 0.0196

Testing k=3...
  Silhouette: 0.6784
  Davies-Bouldin: 0.3319
  Calinski-Harabasz: 605430.8998
  ARI: 0.0191

Testing k=4...
  Silhouette: 0.5489
  Davies-Bouldin: 0.6598
  Calinski-Harabasz: 543214.0442
  ARI: 0.0440

Testing k=5...
  Silhouette: 0.5584
  Davies-Bouldin: 0.5985
  Calinski-Harabasz: 493202.8056
  ARI: 0.0323

Testing k=6...


KeyboardInterrupt: 

# Bayesian Search

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    silhouette_score, 
    davies_bouldin_score, 
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    fowlkes_mallows_score
)
from sklearn.decomposition import PCA
from sklearn_extra.cluster import KMedoids
from skopt import gp_minimize
from skopt.space import Integer, Categorical
from skopt.utils import use_named_args
from skopt.plots import plot_convergence, plot_objective
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Get the Code directory (project root)
current_dir = Path.cwd()
code_dir = current_dir.parent.parent.parent.parent
print(f"Code directory: {code_dir}")

# Define all data paths
PATHS = {
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

def load_all_data():
    """Load all data files"""
    data = {}
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
    
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
    
    return data

def bayesian_optimization_kmedoids(X, y_true, n_calls=30, max_samples=5000):
    """Perform Bayesian Optimization to find optimal K-Medoids parameters"""
    print("\n" + "="*50)
    print("BAYESIAN OPTIMIZATION FOR K-MEDOIDS")
    print("="*50)
    
    # Sample data if too large
    if len(X) > max_samples:
        sample_idx = np.random.choice(len(X), max_samples, replace=False)
        X_sample = X.iloc[sample_idx] if isinstance(X, pd.DataFrame) else X[sample_idx]
        y_sample = y_true.iloc[sample_idx] if isinstance(y_true, pd.Series) else y_true[sample_idx]
    else:
        X_sample = X
        y_sample = y_true
    
    # Define search space
    search_space = [
        Integer(2, 15, name='n_clusters'),
        Categorical(['euclidean', 'manhattan', 'cosine'], name='metric'),
        Categorical(['pam', 'alternate'], name='method'),
        Categorical(['random', 'heuristic', 'k-medoids++'], name='init'),
        Integer(100, 500, name='max_iter')
    ]
    
    # Store results
    iteration_results = []
    
    @use_named_args(search_space)
    def objective(**params):
        """Objective function to minimize (negative silhouette score)"""
        try:
            kmedoids = KMedoids(
                n_clusters=params['n_clusters'],
                metric=params['metric'],
                method=params['method'],
                init=params['init'],
                max_iter=params['max_iter'],
                random_state=42
            )
            
            labels = kmedoids.fit_predict(X_sample)
            
            # Calculate composite score (weighted combination)
            silhouette = silhouette_score(X_sample, labels)
            davies_bouldin = davies_bouldin_score(X_sample, labels)
            calinski = calinski_harabasz_score(X_sample, labels)
            
            # Normalize and combine (maximize silhouette, minimize DB, maximize CH)
            score = silhouette - (davies_bouldin / 10) + (calinski / 10000)
            
            # Store iteration results
            result = {
                'iteration': len(iteration_results) + 1,
                'n_clusters': params['n_clusters'],
                'metric': params['metric'],
                'method': params['method'],
                'init': params['init'],
                'max_iter': params['max_iter'],
                'silhouette': silhouette,
                'davies_bouldin': davies_bouldin,
                'calinski_harabasz': calinski,
                'composite_score': score
            }
            iteration_results.append(result)
            
            print(f"\nIteration {len(iteration_results)}:")
            print(f"  n_clusters={params['n_clusters']}, metric={params['metric']}, "
                  f"method={params['method']}, init={params['init']}")
            print(f"  Silhouette: {silhouette:.4f}, DB: {davies_bouldin:.4f}, "
                  f"CH: {calinski:.2f}")
            print(f"  Composite Score: {score:.4f}")
            
            # Return negative score for minimization
            return -score
            
        except Exception as e:
            print(f"Error in iteration: {e}")
            return 1000  # Large penalty for failed iterations
    
    # Run Bayesian Optimization
    print(f"\nStarting Bayesian Optimization with {n_calls} iterations...")
    result = gp_minimize(
        objective,
        search_space,
        n_calls=n_calls,
        random_state=42,
        verbose=False,
        n_initial_points=10
    )
    
    # Best parameters
    best_params = {
        'n_clusters': result.x[0],
        'metric': result.x[1],
        'method': result.x[2],
        'init': result.x[3],
        'max_iter': result.x[4]
    }
    
    print("\n" + "="*50)
    print("BEST PARAMETERS FOUND:")
    print("="*50)
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    print(f"  Best Composite Score: {-result.fun:.4f}")
    
    return result, best_params, pd.DataFrame(iteration_results)

def plot_bayesian_optimization_results(result, iteration_df, dataset_name, save_path=None):
    """Plot Bayesian Optimization results"""
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    fig.suptitle(f'Bayesian Optimization Results - {dataset_name}', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Convergence plot
    ax1 = fig.add_subplot(gs[0, :2])
    best_scores = np.minimum.accumulate(-iteration_df['composite_score'])
    ax1.plot(iteration_df['iteration'], -iteration_df['composite_score'], 
             'o-', alpha=0.6, label='Iteration Score')
    ax1.plot(iteration_df['iteration'], best_scores, 
             'r-', linewidth=2, label='Best Score So Far')
    ax1.set_xlabel('Iteration', fontsize=11)
    ax1.set_ylabel('Composite Score', fontsize=11)
    ax1.set_title('Optimization Convergence', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Plot 2: Score distribution
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(-iteration_df['composite_score'], bins=20, color='steelblue', 
             edgecolor='black', alpha=0.7)
    ax2.axvline(-iteration_df['composite_score'].min(), color='red', 
                linestyle='--', linewidth=2, label='Best')
    ax2.set_xlabel('Composite Score', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Score Distribution', fontsize=12, fontweight='bold')
    ax2.legend()
    
    # Plot 3: n_clusters vs Score
    ax3 = fig.add_subplot(gs[1, 0])
    scatter = ax3.scatter(iteration_df['n_clusters'], -iteration_df['composite_score'],
                         c=iteration_df['iteration'], cmap='viridis', s=100, alpha=0.6)
    ax3.set_xlabel('Number of Clusters', fontsize=11)
    ax3.set_ylabel('Composite Score', fontsize=11)
    ax3.set_title('Clusters vs Score', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax3, label='Iteration')
    ax3.grid(alpha=0.3)
    
    # Plot 4: Metric comparison
    ax4 = fig.add_subplot(gs[1, 1])
    metric_scores = iteration_df.groupby('metric')[['silhouette', 'davies_bouldin']].mean()
    x = np.arange(len(metric_scores))
    width = 0.35
    ax4.bar(x - width/2, metric_scores['silhouette'], width, label='Silhouette', alpha=0.8)
    ax4_twin = ax4.twinx()
    ax4_twin.bar(x + width/2, metric_scores['davies_bouldin'], width, 
                 label='Davies-Bouldin', alpha=0.8, color='orange')
    ax4.set_xlabel('Distance Metric', fontsize=11)
    ax4.set_ylabel('Silhouette Score', fontsize=11)
    ax4_twin.set_ylabel('Davies-Bouldin Index', fontsize=11)
    ax4.set_title('Metric Comparison', fontsize=12, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(metric_scores.index, rotation=15)
    ax4.legend(loc='upper left')
    ax4_twin.legend(loc='upper right')
    
    # Plot 5: Method comparison
    ax5 = fig.add_subplot(gs[1, 2])
    method_scores = iteration_df.groupby('method')['composite_score'].mean()
    colors = ['#2ecc71', '#e74c3c']
    ax5.barh(range(len(method_scores)), -method_scores, color=colors, alpha=0.8)
    ax5.set_yticks(range(len(method_scores)))
    ax5.set_yticklabels(method_scores.index)
    ax5.set_xlabel('Average Composite Score', fontsize=11)
    ax5.set_title('Method Comparison', fontsize=12, fontweight='bold')
    ax5.grid(axis='x', alpha=0.3)
    
    # Plot 6: Initialization comparison
    ax6 = fig.add_subplot(gs[2, 0])
    init_scores = iteration_df.groupby('init')['composite_score'].mean()
    ax6.bar(range(len(init_scores)), -init_scores, color='purple', alpha=0.7)
    ax6.set_xticks(range(len(init_scores)))
    ax6.set_xticklabels(init_scores.index, rotation=15, ha='right')
    ax6.set_ylabel('Average Composite Score', fontsize=11)
    ax6.set_title('Initialization Strategy Comparison', fontsize=12, fontweight='bold')
    ax6.grid(axis='y', alpha=0.3)
    
    # Plot 7: Top 10 configurations
    ax7 = fig.add_subplot(gs[2, 1:])
    top_10 = iteration_df.nlargest(10, 'composite_score')
    y_pos = np.arange(len(top_10))
    config_labels = [f"k={row['n_clusters']}, {row['metric'][:3]}, {row['method'][:3]}" 
                    for _, row in top_10.iterrows()]
    bars = ax7.barh(y_pos, -top_10['composite_score'], color='teal', alpha=0.7)
    ax7.set_yticks(y_pos)
    ax7.set_yticklabels(config_labels, fontsize=9)
    ax7.set_xlabel('Composite Score', fontsize=11)
    ax7.set_title('Top 10 Configurations', fontsize=12, fontweight='bold')
    ax7.grid(axis='x', alpha=0.3)
    
    # Add values on bars
    for i, (bar, score) in enumerate(zip(bars, -top_10['composite_score'])):
        ax7.text(score, bar.get_y() + bar.get_height()/2, 
                f'{score:.4f}', va='center', ha='left', fontsize=8)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def find_optimal_k(X, y_true, k_range=range(2, 11), method='clara', max_samples=5000):
    """Find optimal number of clusters using various metrics"""
    print("\n" + "="*50)
    print(f"FINDING OPTIMAL K (method={method})")
    print("="*50)
    
    if len(X) > max_samples:
        sample_idx = np.random.choice(len(X), max_samples, replace=False)
        X_sample = X.iloc[sample_idx] if isinstance(X, pd.DataFrame) else X[sample_idx]
        y_sample = y_true.iloc[sample_idx] if isinstance(y_true, pd.Series) else y_true[sample_idx]
    else:
        X_sample = X
        y_sample = y_true
    
    metrics = {
        'k': [],
        'silhouette': [],
        'davies_bouldin': [],
        'calinski_harabasz': [],
        'inertia': [],
        'ari': [],
        'nmi': [],
        'fmi': []
    }
    
    for k in k_range:
        print(f"\nTesting k={k}...")
        
        kmedoids = KMedoids(
            n_clusters=k,
            metric='euclidean',
            method='pam',
            init='k-medoids++',
            max_iter=300,
            random_state=42
        )
        
        labels = kmedoids.fit_predict(X_sample)
        
        metrics['k'].append(k)
        metrics['silhouette'].append(silhouette_score(X_sample, labels))
        metrics['davies_bouldin'].append(davies_bouldin_score(X_sample, labels))
        metrics['calinski_harabasz'].append(calinski_harabasz_score(X_sample, labels))
        metrics['inertia'].append(kmedoids.inertia_)
        metrics['ari'].append(adjusted_rand_score(y_sample, labels))
        metrics['nmi'].append(normalized_mutual_info_score(y_sample, labels))
        metrics['fmi'].append(fowlkes_mallows_score(y_sample, labels))
        
        print(f"  Silhouette: {metrics['silhouette'][-1]:.4f}")
        print(f"  Davies-Bouldin: {metrics['davies_bouldin'][-1]:.4f}")
        print(f"  Calinski-Harabasz: {metrics['calinski_harabasz'][-1]:.4f}")
        print(f"  ARI: {metrics['ari'][-1]:.4f}")
    
    return pd.DataFrame(metrics)

def plot_metrics(metrics_df, save_path=None):
    """Plot clustering metrics for different k values"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('K-Medoids Clustering Metrics vs Number of Clusters', fontsize=16, fontweight='bold')
    
    plot_configs = [
        ('silhouette', 'Silhouette Score', 'Higher is better', 'green'),
        ('davies_bouldin', 'Davies-Bouldin Index', 'Lower is better', 'red'),
        ('calinski_harabasz', 'Calinski-Harabasz Score', 'Higher is better', 'blue'),
        ('inertia', 'Inertia', 'Lower is better', 'orange'),
        ('ari', 'Adjusted Rand Index', 'Higher is better', 'purple'),
        ('nmi', 'Normalized Mutual Info', 'Higher is better', 'cyan'),
        ('fmi', 'Fowlkes-Mallows Index', 'Higher is better', 'magenta'),
    ]
    
    for idx, (metric, title, subtitle, color) in enumerate(plot_configs):
        ax = axes[idx // 4, idx % 4]
        ax.plot(metrics_df['k'], metrics_df[metric], marker='o', linewidth=2, 
                markersize=8, color=color, label=metric)
        ax.set_xlabel('Number of Clusters (k)', fontsize=10)
        ax.set_ylabel(title, fontsize=10)
        ax.set_title(f'{title}\n({subtitle})', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xticks(metrics_df['k'])
        
        if 'lower' in subtitle.lower():
            optimal_idx = metrics_df[metric].idxmin()
        else:
            optimal_idx = metrics_df[metric].idxmax()
        optimal_k = metrics_df.loc[optimal_idx, 'k']
        optimal_val = metrics_df.loc[optimal_idx, metric]
        ax.scatter([optimal_k], [optimal_val], color='red', s=200, marker='*', 
                  zorder=5, label=f'Optimal k={optimal_k}')
        ax.legend(loc='best', fontsize=8)
    
    fig.delaxes(axes[1, 3])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def perform_clustering(X, y_true, n_clusters, metric='euclidean', method='pam', 
                       init='k-medoids++', max_iter=300, dataset_name='Dataset'):
    """Perform K-Medoids clustering and evaluate"""
    print("\n" + "="*50)
    print(f"CLUSTERING: {dataset_name} (k={n_clusters})")
    print("="*50)
    
    kmedoids = KMedoids(
        n_clusters=n_clusters,
        metric=metric,
        method=method,
        init=init,
        max_iter=max_iter,
        random_state=42
    )
    
    labels = kmedoids.fit_predict(X)
    
    metrics = {
        'Silhouette Score': silhouette_score(X, labels),
        'Davies-Bouldin Index': davies_bouldin_score(X, labels),
        'Calinski-Harabasz Score': calinski_harabasz_score(X, labels),
        'Inertia': kmedoids.inertia_,
        'Adjusted Rand Index': adjusted_rand_score(y_true, labels),
        'Normalized Mutual Info': normalized_mutual_info_score(y_true, labels),
        'Fowlkes-Mallows Score': fowlkes_mallows_score(y_true, labels)
    }
    
    print("\nClustering Metrics:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")
    
    return kmedoids, labels, metrics

def visualize_clusters(X, labels, y_true, medoids_idx, dataset_name='Dataset', save_path=None):
    """Visualize clusters using PCA"""
    print(f"\nGenerating visualizations for {dataset_name}...")
    
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'K-Medoids Clustering Visualization - {dataset_name}', 
                 fontsize=16, fontweight='bold')
    
    scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, 
                               cmap='viridis', alpha=0.6, s=30)
    axes[0].scatter(X_pca[medoids_idx, 0], X_pca[medoids_idx, 1], 
                   c='red', marker='X', s=300, edgecolors='black', 
                   linewidths=2, label='Medoids')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    axes[0].set_title('Predicted Clusters', fontsize=12, fontweight='bold')
    axes[0].legend()
    plt.colorbar(scatter1, ax=axes[0], label='Cluster')
    
    scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, 
                               cmap='plasma', alpha=0.6, s=30)
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11)
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11)
    axes[1].set_title('True Labels', fontsize=12, fontweight='bold')
    plt.colorbar(scatter2, ax=axes[1], label='Class')
    
    unique_labels, counts = np.unique(labels, return_counts=True)
    axes[2].bar(unique_labels, counts, color='steelblue', edgecolor='black')
    axes[2].set_xlabel('Cluster', fontsize=11)
    axes[2].set_ylabel('Number of Samples', fontsize=11)
    axes[2].set_title('Cluster Sizes', fontsize=12, fontweight='bold')
    axes[2].set_xticks(unique_labels)
    for i, (label, count) in enumerate(zip(unique_labels, counts)):
        axes[2].text(label, count, str(count), ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def compare_datasets(all_results):
    """Compare clustering results across datasets"""
    print("\n" + "="*50)
    print("COMPARING DATASETS")
    print("="*50)
    
    comparison = pd.DataFrame(all_results).T
    print("\n", comparison)
    
    metrics_to_plot = ['Silhouette Score', 'Davies-Bouldin Index', 
                       'Calinski-Harabasz Score', 'Adjusted Rand Index']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Clustering Performance Comparison Across Datasets', 
                 fontsize=16, fontweight='bold')
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        values = comparison[metric]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        bars = ax.bar(range(len(values)), values, color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xlabel('Dataset', fontsize=11)
        ax.set_ylabel(metric, fontsize=11)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.set_xticks(range(len(values)))
        ax.set_xticklabels(values.index, rotation=15, ha='right')
        ax.grid(axis='y', alpha=0.3)
        
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('clustering_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# Main execution
if __name__ == "__main__":
    data = load_all_data()
    
    if not data:
        print("No data loaded. Exiting.")
        exit()
    
    datasets = [
        ('X_train_smote', 'y_train_smote', 'SMOTE'),
        ('X_train_tomek', 'y_train_tomek', 'Tomek'),
        ('X_train_smote_tomek', 'y_train_smote_tomek', 'SMOTE-Tomek')
    ]
    
    all_results = {}
    bayesian_results = {}
    
    for X_key, y_key, name in datasets:
        if X_key in data and y_key in data:
            X = data[X_key]
            y = data[y_key]
            
            print("\n" + "#"*60)
            print(f"# PROCESSING DATASET: {name}")
            print("#"*60)
            
            # 1. Bayesian Optimization
            bo_result, best_params, iteration_df = bayesian_optimization_kmedoids(
                X, y, n_calls=30, max_samples=5000
            )
            
            # Plot Bayesian Optimization results
            plot_bayesian_optimization_results(
                bo_result, iteration_df, name, 
                save_path=f'bayesian_opt_{name.lower()}.png'
            )
            
            bayesian_results[name] = {
                'best_params': best_params,
                'best_score': -bo_result.fun,
                'iterations_df': iteration_df
            }
            
            # 2. Grid search for optimal k (traditional method)
            metrics_df = find_optimal_k(X, y, k_range=range(2, 8))
            plot_metrics(metrics_df, save_path=f'metrics_{name.lower()}.png')
            
            # 3. Perform clustering with Bayesian-optimized parameters
            model, labels, metrics = perform_clustering(
                X, y, 
                n_clusters=best_params['n_clusters'],
                metric=best_params['metric'],
                method=best_params['method'],
                init=best_params['init'],
                max_iter=best_params['max_iter'],
                dataset_name=f"{name} (Bayesian Optimized)"
            )
            
            # Visualize
            visualize_clusters(X, labels, y, model.medoid_indices_, 
                             f"{name} (Bayesian Optimized)", 
                             save_path=f'clusters_bo_{name.lower()}.png')
            
            all_results[name] = metrics
    
    # Compare all datasets
    if all_results:
        compare_datasets(all_results)
    
    # Summary of Bayesian Optimization
    print("\n" + "="*60)
    print("BAYESIAN OPTIMIZATION SUMMARY")
    print("="*60)
    for name, result in bayesian_results.items():
        print(f"\n{name}:")
        print(f"  Best Parameters: {result['best_params']}")
        print(f"  Best Score: {result['best_score']:.4f}")
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)